In [0]:
drop table if exists cl_mp_de.`03_gold_fact`.fact_retail_analytics

In [0]:
CREATE TABLE cl_mp_de.`03_gold_fact`.fact_retail_analytics (
    fact_key BIGINT GENERATED ALWAYS AS IDENTITY,

    date_key INT,
    customer_key BIGINT,
    product_key BIGINT,
    store_key BIGINT,

    transaction_id STRING,

    quantity_sold INT,
    unit_price DOUBLE,
    base_cost DOUBLE,
    marked_price DOUBLE,

    total_sales DOUBLE,        -- quantity * unit_price
    total_cost DOUBLE,         -- quantity * base_cost
    profit DOUBLE,             -- total_sales - total_cost
    discount_amount DOUBLE,    -- (marked_price - unit_price) * quantity

    return_flag INT,           -- 1 if returned else 0
    stock_on_hand INT          -- latest inventory snapshot

)

In [0]:
-- silver_sales_details → sales
-- silver_mst_product_master → cost & price
-- silver_return_transaction_logs → return_flag
-- silver_inventment_levels → stock
SELECT
    CAST(date_format(ssd.date, 'yyyyMMdd') as INT) as date_key,
    COALESCE(dc.customer_key, 0) AS customer_key,
    dp.product_key,
    ds.store_key,
    ssd.transaction_id,
    ssd.quantity_sold,
    ssd.unit_price,
    dp.base_cost,
    dp.marked_price,
    ssd.quantity_sold * ssd.unit_price AS total_sales,
    ssd.quantity_sold * dp.base_cost AS total_cost,
    total_sales - total_cost AS profit,
    (dp.marked_price - ssd.unit_price) * ssd.quantity_sold as discount_amount,
    CASE 
      WHEN rtl.original_transaction_id IS NULL THEN 0
      ELSE 1
    END as return_flag,
    COALESCE(sil.stock_on_hand, 0) AS stock_on_hand

FROM cl_mp_de.`02_silver`.silver_sales ssd
LEFT JOIN cl_mp_de.`03_gold_dim`.dim_customer dc
  ON ssd.customer_id = dc.customer_id
LEFT JOIN cl_mp_de.`03_gold_dim`.dim_product dp
  ON ssd.product_id = dp.product_id
LEFT JOIN cl_mp_de.`03_gold_dim`.dim_store ds
  ON ssd.store_id = ds.store_id
LEFT JOIN cl_mp_de.`02_silver`.silver_return_transaction rtl
  ON ssd.transaction_id = rtl.original_transaction_id
LEFT JOIN cl_mp_de.`02_silver`.silver_inventory sil
  ON ssd.store_id = sil.store_id AND ssd.product_id = sil.sku_id
ORDER BY date_key, customer_key, product_key, store_key

In [0]:
INSERT INTO cl_mp_de.`03_gold_fact`.fact_retail_analytics (
    date_key,
    customer_key,
    product_key,
    store_key,
    transaction_id,
    quantity_sold,
    unit_price,
    base_cost,
    marked_price,
    total_sales,
    total_cost,
    profit,
    discount_amount,
    return_flag ,
    stock_on_hand
)
SELECT
    CAST(date_format(ssd.date, 'yyyyMMdd') as INT) as date_key,
    COALESCE(dc.customer_key, 0) AS customer_key,
    dp.product_key,
    ds.store_key,
    ssd.transaction_id,
    ssd.quantity_sold,
    ssd.unit_price,
    dp.base_cost,
    dp.marked_price,
    ssd.quantity_sold * ssd.unit_price AS total_sales,
    ssd.quantity_sold * dp.base_cost AS total_cost,
    total_sales - total_cost AS profit,
    (dp.marked_price - ssd.unit_price) * ssd.quantity_sold as discount_amount,
    CASE 
      WHEN rtl.original_transaction_id IS NULL THEN 0
      ELSE 1
    END as return_flag,
    COALESCE(sil.stock_on_hand, 0) AS stock_on_hand

FROM cl_mp_de.`02_silver`.silver_sales ssd

LEFT JOIN cl_mp_de.`03_gold_dim`.dim_customer dc
  ON ssd.customer_id = dc.customer_id

LEFT JOIN cl_mp_de.`03_gold_dim`.dim_product dp
  ON ssd.product_id = dp.product_id
  
LEFT JOIN cl_mp_de.`03_gold_dim`.dim_store ds
  ON ssd.store_id = ds.store_id

LEFT JOIN cl_mp_de.`02_silver`.silver_return_transaction rtl
  ON ssd.transaction_id = rtl.original_transaction_id

LEFT JOIN cl_mp_de.`02_silver`.silver_inventory sil
  ON ssd.store_id = sil.store_id AND ssd.product_id = sil.sku_id
  
ORDER BY date_key, customer_key, product_key, store_key

In [0]:
SELECT * FROM cl_mp_de.`03_gold_fact`.fact_retail_analytics